In [ ]:
# ===== Standard library =====
import csv
import datetime
import glob
import os
import random
import re
from collections import Counter
from datetime import timedelta

# ===== Numerical & data handling =====
import numpy as np
import pandas as pd
import polars as pl
from joblib import dump, load
from numpy.random import default_rng

# ===== Visualization =====
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# ===== Scikit-learn =====
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    auc,
    average_precision_score,
    classification_report,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GroupShuffleSplit, KFold, train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# ===== Imbalanced-learn =====
from imblearn.under_sampling import RandomUnderSampler

# ===== TensorFlow / Keras =====
import tensorflow as tf
from keras.utils import to_categorical
from tensorflow import keras
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import (
    Activation,
    Add,
    BatchNormalization,
    Concatenate,
    Conv1D,
    Dense,
    Dropout,
    Embedding,
    Flatten,
    GlobalAveragePooling1D,
    Input,
    MaxPooling1D,
)
from tensorflow.keras.models import Model, Sequential

# ===== Project-specific modules =====
import utils.data_models as data_models
import utils.data_preprocessing as data_preprocessing
from utils.data_integration import combine_data, read_data

In [ ]:
# convert all sublists to arrays
def flatten(subject):
    array_data = [np.array(x) for x in subject]  
    flattened_data = np.concatenate(array_data, axis=0)
    return flattened_data

## Read data and assing age groups

In [ ]:
"""
    Calls the "read_data()" function from file "data_integration.py"
    Calls the "combine_data()" function from file "data_integration.py"
    Parameter:
    - 1: combines only glucose data
    - 2: combines glucose and dempgraphics
    - 3: combines glucose and heartrate
    Output: This function returns the intrgrated dataset included the restricted databases if downloaded and added to folder "datasets for T1D"
 """
# reads all datasets, if they are stored in the required folder Data/datasets_for_T1D...
# DiaData includes two restricted datasets, these should be downloaded from the publishers and shored in the folder
restricted_list_5 = read_data(read_all = False, frequency = "15min")
# integrates the restricted datasets with modus = 2
df_5min = combine_data(2, restricted_list_5, columns_to_check=["Age", "Sex"])

# combine with not restricted data
# This dataset can be downloaded from https://zenodo.org/records/16875703 and should be stored in Data/datasets_for_T1D 
sdb_I_5min = pl.read_csv("Data/datasets_for_T1D/MDB.csv")
sdb_I_5min  = sdb_I_5min.sort(["PtID", "ts"])
sdb_I_5min = sdb_I_5min.to_pandas()
# converts the timestamp column to datetime
sdb_I_5min["ts"] = pd.to_datetime(sdb_I_5min["ts"])

# combines both dataframes to have the final integrated dataset 
df_5min = pd.concat([df_5min, sdb_I_5min])

In [ ]:
result_ts = df_5min.with_columns(
    pl.col("ts").str.to_datetime()
)

df_sorted = result_ts.sort(["PtID", "ts"])
df_all = df_sorted.unique(subset=["PtID", "ts"], keep="first")

n_subjects = df_all.select(pl.col("PtID").n_unique()).item()
print(n_subjects)

In [ ]:
cleaned_data = data_preprocessing.remove_outliers_polars(df_all, value="GlucoseCGM")
data_cleanedpd = cleaned_data.to_pandas()
data_cleanedpd = data_cleanedpd.sort_values(by=["PtID", "ts"]).reset_index(drop=True)

# linear interpolation is used for gaps which are less than 30 minute (6 consecuitve datapoints)
data_cleanedpd["GlucoseCGM"] = data_cleanedpd.groupby("PtID")["GlucoseCGM"].transform(
    lambda x: data_preprocessing.gap_limited_interpolation(x, limit=6)
)

# stineman interpolation for values between 30 adn 120 minutes
df_interpolated_stine = data_cleanedpd.groupby("PtID", group_keys=False).apply(lambda x: data_preprocessing.interpolate_stineman_group(x, timestamp = "ts", value = "GlucoseCGM", llimit=6, ulimit=24))
df_interpolated_stine = df_interpolated_stine.sort_index()

df_interpolated_stine = df_interpolated_stine[["ts", "GlucoseCGM", "PtID"]]
df_interpolated_stine

## class assgnment

In [ ]:
# generates classes for the hypoglycemia classification task
def class_generation(df_copy: pd.DataFrame, timestamp_col: str, start: int, end: int, class_number: int, col_name: str) -> pl.DataFrame: 
    """
    This function is used to assign labels.
    Parameters:
   - df_copy is the original pandas dataframe, 
   - timestamp_col is the name of the column with the timestamps, 
   - start is time in minutes of the minimum duration before hypogylcemia
   - end is time in minutes of the maximum duration before hypogylcemia
   - class_number is the class for the defined time range before hypoglycemia
    Output: It returns the dataframe with the assigned class for the spefied time range before hypoglycemia.
   """ 
    # first, the dataframe is converted to polars to increase efficacy
    df = pl.from_pandas(df_copy)
    # timestamps are sorted 
    df = df.sort(timestamp_col)
    
    # zero events are hypoglycemic data points
    # these are selected and the timestamps are stores as a series
    event_times = df.filter(pl.col(col_name) == 0).select(timestamp_col).to_series()

    # the start and end time of the time range before hypogylcemia are computed
    start_bounds = event_times - datetime.timedelta(minutes=start)
    end_bounds = event_times - datetime.timedelta(minutes=end)

    # the series is masked 
    mask = pl.Series([False] * df.height)

    # timestamps meeting the defined criteria of specified time bonds are identified 
    # moreover, the class should be -1 (not assinged as other classes before)
    for start_time, end_time in zip(start_bounds, end_bounds):
        window_mask = (
            (df[timestamp_col] > end_time) &
            (df[timestamp_col] <= start_time) &
            (df[col_name] == -1)
        )
        mask = mask | window_mask

    df = df.with_columns([
        pl.when(mask).then(class_number).otherwise(pl.col(col_name)).alias(col_name)
    ])

    # the dataframe is converted back to pandas 
    df_pd = df.to_pandas()
    # the dataframe with assigned classes is returned
    return df_pd

In [ ]:
#test_data_classes = []
#for i in test_data_cleaned:
classes_df = df_interpolated_stine.sort_values(by=["PtID", "ts"]).reset_index(drop=True)
                                                    
# initially all classes are assigned a class -1
classes_df["Class1"] = -1

# hypoglycemic values are defined as class 0 with values equal and lower than 70 mg/dL
classes_df.loc[classes_df["GlucoseCGM"] <= 70, "Class1"] = 0

# calls the function class_generation to assing classes based on wanted intervals before hypogylcemia for each subject separately
classes_df = classes_df.groupby("PtID", group_keys=False).apply(lambda x: class_generation(x,"ts", 5, 20, 1, col_name = "Class1")) 
classes_df = classes_df.groupby("PtID", group_keys=False).apply(lambda x: class_generation(x,"ts", 20, 50, 2, col_name = "Class1"))  
classes_df = classes_df.groupby("PtID", group_keys=False).apply(lambda x: class_generation(x,"ts", 50, 125, 3, col_name = "Class1"))  
classes_df.loc[classes_df["Class1"] == -1, "Class1"] = 4

# sorts the glucose values by patient id and timestamps
classes_df = classes_df.sort_values(by=["PtID", "ts"])
# reindexes based on the sorted dataset
classes_df = classes_df.reset_index(drop=True)
# nneu noch mehr neu
print(classes_df["Class1"].value_counts())

In [ ]:
# initially all classes are assigned a class -1
classes_df["Class1"] = -1

# hypoglycemic values are defined as class 0 with values equal and lower than 70 mg/dL
classes_df.loc[classes_df["GlucoseCGM"] <= 70, "Class1"] = 0

# calls the function class_generation to assing classes based on wanted intervals before hypogylcemia for each subject separately
classes_df = classes_df.groupby("PtID", group_keys=False).apply(lambda x: class_generation(x,"ts", 5, 20, 1, col_name = "Class1")) # 5-15 min
classes_df = classes_df.groupby("PtID", group_keys=False).apply(lambda x: class_generation(x,"ts", 20, 50, 2, col_name = "Class1")) # 15-30 min
classes_df = classes_df.groupby("PtID", group_keys=False).apply(lambda x: class_generation(x,"ts", 50, 125, 3, col_name = "Class1"))  # 30-60 min
classes_df.loc[classes_df["Class1"] == -1, "Class1"] = 4

# sorts the glucose values by patient id and timestamps
classes_df = classes_df.sort_values(by=["PtID", "ts"])
# reindexes based on the sorted dataset
classes_df = classes_df.reset_index(drop=True)
# nneu noch mehr neu
print(classes_df["Class1"].value_counts())
pl_df = pl.from_pandas(df_sorted)

# Save to CSV
pl_df.write_csv("Data/df_classes.csv")

## features

In [ ]:
df_polars = pl.from_pandas(df_sorted)
df_polars = df_polars.sort(["PtID", "ts"])
df_polars = df_polars.unique(subset=["PtID", "ts"], keep="first")
# Read CSV with schema
df_polars = pl.read_csv("Data/df_classes.csv")

df_polars = df_polars.with_columns(
    pl.col("ts").str.to_datetime()
)

df_polars = df_polars.sort(["PtID", "ts"])

# Apply per SubjectID
df_features = (
    df_polars.group_by("PtID", maintain_order=True)
    .map_groups(data_preprocessing.add_time_features)) 

df_features = df_features.drop_nulls(["GlucoseCGM"])

In [ ]:
def apply_all_metrics_lazy_per_patient(
    lf: pl.LazyFrame,
    pt_col: str = "PtID",
    ts_col: str = "ts",
    column: str = "GlucoseCGM",
    sample_every: str = "5m",
    windows: Sequence[tuple[str, str]] = (("30m", "30min"), ("1h", "1hr"), ("2h", "2hr"), ("4h", "4hr"),("8h", "8hr"),("24h", "24hr")),
) -> pl.LazyFrame:
    lf = (
        lf.with_columns(pl.col(ts_col).cast(pl.Datetime))
          .sort([pt_col, ts_col])
    )

    def to_minutes(s: str) -> int:
        s = s.strip().lower()
        if s.endswith("m"):
            return int(s[:-1])
        if s.endswith("h"):
            return int(s[:-1]) * 60
        raise ValueError(f"Unsupported duration: {s} (use 'm' or 'h')")

    samp_min = to_minutes(sample_every)
    samp_dur = pl.duration(minutes=samp_min)

    for window_size, suffix in windows:
        win_min = to_minutes(window_size)
        if win_min % samp_min != 0:
            raise ValueError(f"window_size {window_size} must be a multiple of sample_every {sample_every}")

        expected_n = win_min // samp_min
        expected_span = pl.duration(minutes=samp_min * (expected_n - 1))

        # temp column names per window
        gap_col = f"__gap_{suffix}"
        span_col = f"__span_{suffix}"
        n_col = f"__n_{suffix}"
        ok_col = f"__ok_{suffix}"

        ts = pl.col(ts_col)
        c = pl.col(column).cast(pl.Float64)

        # 1) create helper columns FIRST (no nesting window->rolling)
        lf = lf.with_columns([
            ts.diff().over(pt_col).alias(gap_col),
            (ts - ts.shift(expected_n - 1)).over(pt_col).alias(span_col),

            # count non-null samples in trailing time window
            c.is_not_null()
             .cast(pl.Int32)
             .rolling_sum_by(ts, window_size=window_size, closed="right")
             .over(pt_col)
             .alias(n_col),
        ])

        # 2) compute "window complete & gap-free" mask using helper cols
        lf = lf.with_columns([
            (
                (pl.col(n_col) == expected_n) &
                (pl.col(gap_col).rolling_max(expected_n - 1).over(pt_col) <= samp_dur) &
                (pl.col(span_col) == expected_span)
            ).alias(ok_col)
        ])

        # helper: null out if window isn't ok
        def only_if_ok(expr: pl.Expr) -> pl.Expr:
            return pl.when(pl.col(ok_col)).then(expr).otherwise(None)

        # 3) add metrics (time-based rollings) masked by ok_col
        lf = lf.with_columns([
            only_if_ok(
                c.rolling_mean_by(ts, window_size=window_size, closed="right", min_samples=expected_n).over(pt_col)
            ).alias(f"{column}_mean_{suffix}"),

            only_if_ok(
                c.rolling_std_by(ts, window_size=window_size, closed="right", min_samples=expected_n).over(pt_col)
            ).alias(f"{column}_std_{suffix}"),

            only_if_ok(
                c.rolling_min_by(ts, window_size=window_size, closed="right", min_samples=expected_n).over(pt_col)
            ).alias(f"{column}_min_{suffix}"),

            only_if_ok(
                c.rolling_max_by(ts, window_size=window_size, closed="right", min_samples=expected_n).over(pt_col)
            ).alias(f"{column}_max_{suffix}"),

            # keep a public n_{suffix} too (null when invalid)
            only_if_ok(pl.col(n_col)).alias(f"n_{suffix}"),

            only_if_ok(
                ((c < 70).cast(pl.Int8)
                 .rolling_mean_by(ts, window_size=window_size, closed="right", min_samples=expected_n)
                 .over(pt_col) * 100)
            ).alias(f"time_below70_{suffix}"),

            only_if_ok(
                ((c > 200).cast(pl.Int8)
                 .rolling_mean_by(ts, window_size=window_size, closed="right", min_samples=expected_n)
                 .over(pt_col) * 100)
            ).alias(f"time_above200_{suffix}"),

            only_if_ok((c - c.shift(expected_n - 1)).over(pt_col)).alias(f"diff_{suffix}"),
            only_if_ok(((c - c.shift(expected_n - 1)).over(pt_col)) / win_min).alias(f"slope_{suffix}"),  # per minute
        ])

        # optional: drop temp cols to keep df clean
        lf = lf.drop([gap_col, span_col, n_col, ok_col])

    return lf

In [ ]:
lf_out = apply_all_metrics_lazy_per_patient(df_features.lazy())
df_out = lf_out.collect()
df_clean_out = df_out.drop_nulls(["GlucoseCGM_mean_30min", "GlucoseCGM_mean_1hr",
        "GlucoseCGM_mean_2hr"])
del df_out
del df_polars

In [ ]:
def add_hypo_event_window_metrics_time(
    df: pl.DataFrame,
    pt_col: str = "PtID",
    ts_col: str = "ts",
    glucose_col: str = "GlucoseCGM",
    threshold: float = 70.0,
    sample_minutes: int = 5,
    windows_min: Sequence[int] = (30, 60, 120, 240, 1440),
) -> pl.DataFrame:
    if df.is_empty():
        return df

    df = (
        df.with_columns(pl.col(ts_col).cast(pl.Datetime))
          .sort([pt_col, ts_col])
    )

    x = pl.col(glucose_col).cast(pl.Float64)
    ts = pl.col(ts_col)

    hypo = x.le(threshold).fill_null(False)    # null glucose breaks events (conservative)
    obs  = x.is_not_null()

    samp_dur = pl.duration(minutes=sample_minutes)

    # Sentinel datetime for "no event end in window"
    sentinel_ts = pl.lit(datetime(1900, 1, 1)).cast(pl.Datetime)

    out = (
        df.with_columns([
            hypo.alias("_hypo"),
            obs.alias("_obs"),
            hypo.shift(1).over(pt_col).fill_null(False).alias("_prev_hypo"),
            hypo.shift(-1).over(pt_col).fill_null(False).alias("_next_hypo"),
        ])
        .with_columns([
            (pl.col("_hypo") & ~pl.col("_prev_hypo")).alias("_event_start"),
            (pl.col("_hypo") & ~pl.col("_next_hypo")).alias("_event_end"),
        ])
        .with_columns([
            # event_id only for hypo rows; resets per patient
            pl.when(pl.col("_hypo"))
              .then(pl.col("_event_start").cast(pl.Int64).cum_sum().over(pt_col))
              .otherwise(None)
              .alias("event_id"),
        ])
        .with_columns([
            # duration only meaningful on event-end rows; else 0 (NO NULLS)
            pl.when(pl.col("_event_end"))
              .then(pl.len().over([pt_col, "event_id"]) * sample_minutes)
              .otherwise(0)
              .alias("event_duration_end_min"),

            # event-end timestamp: sentinel on non-end rows (NO NULLS)
            pl.when(pl.col("_event_end"))
              .then(ts)
              .otherwise(sentinel_ts)
              .alias("event_end_ts_filled"),

            # gap helper (per patient)
            ts.diff().over(pt_col).alias("_gap"),
        ])
    )

    exprs: list[pl.Expr] = []

    for w_min in windows_min:
        suf = "24h" if w_min == 1440 else (f"{w_min}m" if w_min < 60 else f"{w_min//60}h")
        window_size = f"{w_min}m" if w_min < 60 else f"{w_min//60}h"

        expected_n = max(1, w_min // sample_minutes)
        expected_span = pl.duration(minutes=sample_minutes * (expected_n - 1))

        # Valid-window checks (subject-wise via .over(pt_col))
        n_obs = (
            pl.col("_obs").cast(pl.Int32)
              .rolling_sum_by(ts, window_size=window_size, closed="right")
              .over(pt_col)
        )
        span_ok = (ts - ts.shift(expected_n - 1)).over(pt_col) == expected_span
        max_gap_ok = (
            pl.col("_gap")
              .rolling_max(expected_n - 1)
              .over(pt_col)
              <= samp_dur
        )
        window_ok = (n_obs == expected_n) & span_ok & max_gap_ok

        def only_if_ok(e: pl.Expr) -> pl.Expr:
            return pl.when(window_ok).then(e).otherwise(None)

        # 1) hypo points in time window
        hypo_points = (
            pl.col("_hypo").cast(pl.Int32)
              .rolling_sum_by(ts, window_size=window_size, closed="right")
              .over(pt_col)
        )
        exprs.append(only_if_ok(hypo_points).alias(f"hypo_points_{suf}"))

        # 2) avg event duration among events that END in the window
        # Avoid rolling_mean_by over nulls -> use sum/count (both non-null)
        dur_sum = (
            pl.col("event_duration_end_min").cast(pl.Float64)
              .rolling_sum_by(ts, window_size=window_size, closed="right")
              .over(pt_col)
        )
        end_count = (
            pl.col("_event_end").cast(pl.Int32)
              .rolling_sum_by(ts, window_size=window_size, closed="right")
              .over(pt_col)
        )
        avg_dur = pl.when(end_count > 0).then(dur_sum / end_count).otherwise(0.0)
        exprs.append(only_if_ok(avg_dur).fill_null(0).alias(f"avg_event_duration_min_{suf}"))

        # 3) minutes since last event END within window
        last_end = (
            pl.col("event_end_ts_filled")
              .rolling_max_by(ts, window_size=window_size, closed="right")
              .over(pt_col)
        )
        last_end_or_null = pl.when(last_end == sentinel_ts).then(None).otherwise(last_end)
        mins_since = (ts - last_end_or_null).dt.total_minutes()
        exprs.append(only_if_ok(mins_since).fill_null(0).alias(f"min_since_last_event_{suf}"))

    return (
        out.with_columns(exprs)
           .drop([
               "_obs", "_hypo", "_prev_hypo", "_next_hypo",
               "_event_start", "_event_end", "event_id",
               "event_duration_end_min", "event_end_ts_filled", "_gap"
           ])
    )


In [ ]:
df_in = df_features.select(["ts", "PtID", "GlucoseCGM"])

df_hypo = add_hypo_event_window_metrics_time(
    df_in,
    pt_col="PtID",
    ts_col="ts",
    glucose_col="GlucoseCGM",
    threshold=70,
    sample_minutes=5,
    windows_min=(30, 60, 120, 240, 1440),
)
del df_features
del df_in
df_hypo = df_hypo.drop("GlucoseCGM")
df_clean_out = df_clean_out.join(df_hypo, on=["PtID", "ts"], how="left")
del df_hypo
df_clean_out =df_clean_out.with_columns([
    pl.col("ts").dt.hour().alias("hour"),
    pl.col("ts").dt.minute().alias("minute"),
    pl.col("ts").dt.weekday().alias("dayofweek")
])
windows = ["30min", "1hr", "2hr", "4hr", "8hr", "24hr"]

df_clean_out = df_clean_out.with_columns([
    pl.col(f"n_{w}").is_not_null().cast(pl.Int8).alias(f"valid_{w}")
    for w in windows
])

In [ ]:
df_imp = df_clean_out.with_columns(
    pl.selectors.numeric().fill_null(0)
)

del df_clean_out

## extract windows

In [ ]:
# Extract min and max from the scaler
x_min = df_imp["GlucoseCGM"].min()
x_max = df_imp["GlucoseCGM"].max()

# Compute scaled threshold
t_scaled = (70 - 40) / (500 - 40)
print("Scaled threshold:", t_scaled)

In [ ]:
def normalize_data(df: pl.DataFrame, value: str, new_col: str) -> pl.DataFrame:
    denom = pl.col(value).max() - pl.col(value).min()
    return df.with_columns(
        pl.when(denom == 0)
        .then(0.0)
        .otherwise((pl.col(value) - pl.col(value).min()) / denom)
        .alias(new_col)
    )
df_imp = normalize_data(df_imp, value="GlucoseCGM", new_col="GlucoseCGM_norm")
#df_imp.write_ipc("data_preprocessed.arrow", compression="zstd")
#df_imp = pl.read_ipc("data_preprocessed.arrow")

In [ ]:
df_imp = df_imp.select(['ts',
 'GlucoseCGM',
 'PtID',
 'Class1',
 'GlucoseCGM_norm'])

## time series generation

In [ ]:
# generates time series with a sliding window appraoch of 2 hour lengths for the maindatabase
# splits data into train, validation, and test
def extract_valid_windows_GLC(
    df_org: pd.DataFrame,
    timestamp_col: str = "ts",
    feature_col: str = "GlucoseCGM_norm",
    class_col: str = "Class",
    expected_sample_count: int = 25,
    min_window_duration=np.timedelta64(2, "h"),
    expected_step=np.timedelta64(5, "m"),  # <-- new: required sampling period
):
    """
    This function is used genearte time series data and splits the data into train, validation, and test data.
    Parameters:
    - df_org is the original pandas dataframe,
    - timestamp_col is the name of the column with the timestamps,
    - feature_col is the name of the column which should be returned as a time series,
    - class_col is name of the column with the classes,
    - expected_sample_count is the number of allowed continuous values in a time series,
    - min_window_duration is the allowed continuous time of a time series
    - expected_step is the expected fixed time delta between consecutive samples (e.g., 5 minutes)
    Output: It returns a list including two separate lists for X_train and Y_train
    """
    # X_train, and Y_train are initialized as empty arrays
    X_train = []
    Y_train = []
   # X_train_feat = []
    valid_ts = []
    all_ts = []


    
    # converts pandas DataFrame to polars for faster execution
    #df = pl.from_pandas(df_org)
    df =  df_org
    # sorts by timestamp to ensure chronological order
    df = df.sort(timestamp_col)
   # features = df.select(pl.exclude(["Class0","Class1", "GlucoseCGM_norm", "PtID", "ts"])).columns
    
    # extracts timestamps and total row count
    timestamps = df[timestamp_col].to_numpy()
    n_rows = df.height

    # initializes empty lists for feature windows and labels
    windows = []
    labels = []
   # windows_feature = []

    # sets starting index for window generation
    start_idx = 0

    # iterates over all rows to construct valid windows
    for end_idx in range(n_rows):
        # moves start index if window exceeds allowed duration
        while timestamps[end_idx] - timestamps[start_idx] > min_window_duration:
            start_idx += 1

        # calculates number of samples in current window
        count = end_idx - start_idx + 1

        # checks if window meets required duration and sample count
        if (
            timestamps[end_idx] - timestamps[start_idx] >= min_window_duration
            and count == expected_sample_count
        ):
            # extracts current time window
            window_df = df.slice(start_idx, count)

            # --- NEW: reject windows with gaps / irregular sampling ---
            w_ts = window_df[timestamp_col].to_numpy()
            # must be exactly expected_step between every consecutive timestamp
            if w_ts.size >= 2:
                diffs = np.diff(w_ts)
                if not np.all(diffs == expected_step):
                    continue
            # ---------------------------------------------------------

            # skips window if missing values are present
            nulls = window_df.null_count().row(0)
            if all(c == 0 for c in nulls):
                # retrieves label from last row in window
                label = window_df[class_col].to_list()[-1]
                # allows only classes 0–4 (your code actually allows 0–3)
                #if label in {0, 1, 2, 3}:
                # stores feature sequence and label
                windows.append(window_df[feature_col].to_list())
                labels.append(label)

                # feat_ids = window_df.select(features).row(-1)   
                # feat_ids = [int(v) for v in feat_ids]
                # windows_feature.append(feat_ids)
                feat_ids = window_df.select("ts").row(-1)   
                #feat_ids = [int(v) for v in feat_ids]
                valid_ts.append(feat_ids)

    # checks if any valid windows were found
    if len(windows) > 0:
        # converts to numpy arrays and reshapes data for model input
        X_data = np.array(windows).reshape(-1, expected_sample_count, 1)
        Y_data = np.array(labels).reshape(-1, 1)
       # X_data_feat = np.array(windows_feature) 

        # appends subject data to training sets
        X_train.append(X_data)
        Y_train.append(Y_data)

        X_data_feat = np.array(valid_ts) 

        all_ts.append(X_data_feat)
       # X_train_feat.append(X_data_feat)
        
        # returns feature and label arrays
        return  X_train, Y_train, all_ts #X_train, Y_train, X_train_feat
    else:
        # returns nothing if no valid window exists
        print("No valid samples generated for this subject!")
        return None
def create_X_Y(df: pl.DataFrame, sample_count=25, hours=2, modus="h", class_c="Class"):
    # split into per-subject DataFrames
    df = df.sort(["PtID", "ts"])
    groups = df.partition_by("PtID", as_dict=True)

    del df

    dic_result = {}
    for ptid, df_person in groups.items():
        out = extract_valid_windows_GLC(df_person,class_col=class_c,
            expected_sample_count=sample_count,
            min_window_duration=np.timedelta64(hours, modus))
        if out is None:
            print(f"WARNING: PtID {ptid} returned None")
            continue  

        X_data, Y_data, X_ts = out
        dic_result[ptid] = {"X": X_data, "Y": Y_data, "X_ts": X_ts}

    return dic_result


In [ ]:
dic_data = create_X_Y(df_imp, 25, 2, 'h', class_c = "Class1")
with gzip.open("Data/timeseries.pkl.gz", "wb") as f:
    pickle.dump(dic_data, f, protocol=pickle.HIGHEST_PROTOCOL)
## combine ts with feature data

with gzip.open("Data/timeseries.pkl.gz", "rb") as f:
   Dic_data = pickle.load(f)

## Filter Patients from Feature Column

In [ ]:
def dict_to_allowed_pairs(d, subject_col="subject_id", ts_col="timestamp"):
    subjects = []
    timestamps = []

    for subj_key, payload in d.items():
        # subj_key is like ('001_D1NAMO',) -> get the string
        subj = subj_key[0] if isinstance(subj_key, tuple) else subj_key

        # payload["ts"] is a list of numpy arrays shaped like (n,1) with datetimes
        for arr in payload.get("X_ts", []):
            # flatten to 1D python list of datetime objects
            ts_list = np.asarray(arr).ravel().tolist()

            subjects.extend([subj] * len(ts_list))
            timestamps.extend(ts_list)

    return pl.DataFrame({subject_col: subjects, ts_col: timestamps})

In [ ]:
keep_df = dict_to_allowed_pairs(Dic_data, subject_col="PtID", ts_col="X_ts")
# If df["timestamp"] is Datetime, align keep_df to it:
keep_df = keep_df.rename({"X_ts": "ts"})
keep_df = keep_df.with_columns(
    pl.col("ts").cast(df_imp.schema["ts"])
)

df_filtered = df_imp.join(keep_df, on=["PtID", "ts"], how="semi")
print(df_imp.height, "->", df_filtered.height)
print(keep_df.head())
del df_imp
del keep_df

# Data Splitting and Storing

In [4]:
X_patients = []
Y_patients = []
subjects = []

for subj_key, payload in Dic_data.items():
    subj = subj_key[0] if isinstance(subj_key, tuple) else subj_key
    chunks_X = payload["X"]  
    chunks_Y = payload["Y"]  


    X_patient = np.concatenate(chunks_X, axis=0)  
    Y_patient = np.concatenate(chunks_Y, axis=0) 

    if X_patient.shape[0] != Y_patient.shape[0]:
        raise ValueError(f"{subj}: X and Y length mismatch: {X_patient.shape[0]} vs {Y_patient.shape[0]}")

    subjects.append(subj)
    X_patients.append(X_patient)
    Y_patients.append(Y_patient) 

In [5]:
del Dic_data

In [6]:
len(subjects)

2509

In [7]:
from sklearn.model_selection import train_test_split
# ensures reproducability
seed_value= 2 
os.environ['PYTHONHASHSEED']=str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
tf.compat.v1.set_random_seed(seed_value)


# assuming X_data and Y_data are lists (or arrays) where each entry corresponds to one subject
subject_indices = list(range(len(subjects)))

# split into temp and test (20%)
train_sub, test_sub = train_test_split(subjects, test_size=0.20, shuffle=True, random_state=seed_value)
train_idx, test_idx = train_test_split(subject_indices, test_size=0.20, shuffle=True, random_state=seed_value)

In [8]:
a= test_sub
b = test_idx
with open("test_subjects.txt", "w") as f:
    f.write("List 1:\n")
    f.write("\n".join(map(str, a)))
    f.write("\n\nList 2:\n")
    f.write("\n".join(map(str, b)))

In [ ]:
def extract_valid_windows_GLC(df_org: pl.DataFrame, label_m:int):
    df =  df_org.sort(["PtID", "ts"])

    # X_train, X_val, X_test, Y_train, Y_val, and Y_test are initialized as empty arrays
    X_data = []

    X = df.drop(["PtID", "ts"]).to_numpy() 
    X_data.append(X)

    return X_data

# generates time series with sliding window approach
df_filtered = df_filtered.sort(["PtID", "ts"])


In [10]:
groups = df_filtered.partition_by("PtID", as_dict=True)
X_data_static = []
del df_filtered

In [11]:
for ptid, df_person in groups.items():
    out = extract_valid_windows_GLC(df_person, label_m=1)
    (
       X_data
    ) = out

    X_data_static.append(X_data)

In [ ]:
dump(X_data_static, "Data/X_static_data.joblib", compress=("zlib", 1))

['X_static_data.joblib']

In [ ]:
X_data_static = load("Data/X_static_data.joblib")

In [ ]:
from sklearn.model_selection import train_test_split
# ensures reproducability
seed_value= 42 
os.environ['PYTHONHASHSEED']=str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
tf.compat.v1.set_random_seed(seed_value)


# extract data by indices
X_train_all = [X_patients[i] for i in train_idx]
Y_train_all = [Y_patients[i] for i in train_idx]

X_test_all = [X_patients[i] for i in test_idx]
Y_test_all = [Y_patients[i] for i in test_idx]

X_train_static_all = [X_data_static[i] for i in train_idx]
X_train_feat_all = [arr for sublist in X_train_static_all for arr in sublist]

X_test_static_all = [X_data_static[i] for i in test_idx]
X_test_feat_all = [arr for sublist in X_test_static_all for arr in sublist]

In [ ]:
print(len(X_train_all))
print(len(X_test_all))
print(len(X_train_feat_all))
print(len(X_test_feat_all))

In [ ]:
import numpy as np

def filter_out_class_5(X_list, Y_list, X_list_feat, drop_value=5):
    X_filt, Y_filt, X_filt_feat = [], [], []

    for X, Y, X_feat in zip(X_list, Y_list, X_list_feat):
        y = np.asarray(Y).reshape(-1)          # (n_i,)
        mask = y != drop_value                 # keep everything except 5

        X_filt.append(np.asarray(X)[mask])
        Y_filt.append(y[mask].reshape(-1, 1))  # keep original (n_i,1) style if you want
        X_filt_feat.append(np.asarray(X_feat)[mask])

    return X_filt, Y_filt, X_filt_feat

X_train, Y_train, X_feat_train = filter_out_class_5(X_train_all, Y_train_all, X_train_feat_all, drop_value=4)
X_test, Y_test, X_feat_test = filter_out_class_5(X_test_all, Y_test_all, X_test_feat_all, drop_value=4)

In [ ]:
for i, (X, Y) in enumerate(zip(X_train, Y_train)):
    assert X.shape[0] == Y.shape[0]
    assert not np.any(Y == 4)

In [ ]:
for i, (X, Y) in enumerate(zip(X_test, Y_test)):
    assert X.shape[0] == Y.shape[0]
    assert not np.any(Y == 4)

In [ ]:
dump((X_train, Y_train), "Data/DL_Train.joblib", compress=("zlib", 1))
dump((X_feat_train, Y_train), "Data/ML_Train.joblib", compress=("zlib", 1))

In [ ]:
dump((X_test, Y_test), "Data/DL_Test.joblib", compress=("zlib", 1))
dump((X_feat_test, Y_test), "Data/ML_Test.joblib", compress=("zlib", 1))